# Flow shop tutorial, Q4: Johnson's rule checked with a mixed-integer program

Tutorial: *Week 7 - Flow Shop Scheduling - Tutorial*, Q4: "Can you write a Linear Program or Mixed-integer
Linear Program to solve this?" This notebook answers it for the data of Q1–Q3.

**The problem.** Every job visits the machines in the same order (A, then B, …). In a *flow shop* every
machine also processes the jobs in the same order, so a schedule is one order of the jobs. Find the order with
the smallest makespan $C_{\max}$ (the time the last job leaves the last machine).

**The model** (the job shop model from the notes, with one ordering variable per *pair of jobs* instead of per
pair of operations):

- $s_{jk} \ge 0$: start time of job $j$ on machine $k$; $p_{jk}$: its processing time;
- $y_{ab} \in \{0,1\}$ for every pair of jobs $a < b$: $y_{ab}=1$ if $a$ goes before $b$ (on every machine);
- minimise $C_{\max}$ subject to
  - $s_{jk} \ge s_{j,k-1} + p_{j,k-1}$ (a job reaches machine $k$ only after machine $k-1$),
  - $s_{bk} \ge s_{ak} + p_{ak} - H(1-y_{ab})$ and $s_{ak} \ge s_{bk} + p_{bk} - H\,y_{ab}$ (no overlap on machine $k$; $H = \sum p_{jk}$),
  - $C_{\max} \ge s_{j,\text{last}} + p_{j,\text{last}}$ for every job.

**How to run it.** Run the first code cell first (it installs Pyomo and the HiGHS solver; the first time takes
about a minute), then *Run → Run All Cells*.

**Expected result.** Q1: makespan **30** (Johnson's order R S P Q T is optimal). Q2: **74** (Johnson's orders
T Q R S P and T P Q R S are optimal). Q3: **64**, for example R P T Q S: better than the 66 of the Campbell,
Dudek and Smith heuristic (R Q P T S) and the 75 of the original order. Where several orders are optimal, the
solver may print a different one with the same makespan.


In [ ]:
# Run this cell first.
# In the browser (JupyterLite) it installs the extra packages into this tab;
# in Colab or Anaconda it installs them with pip (already installed = no change).
%pip install pyomo highspy

import sys
if sys.platform == "emscripten":  # only in the browser (JupyterLite): adapt Pyomo to it
    import contextlib, pyomo.common.tee, pyomo.common.formatting as _fmt
    # 1) browsers have no threads: skip Pyomo's threaded capture of solver output
    pyomo.common.tee.capture_output = lambda *args, **kwargs: contextlib.nullcontext()
    for _m in ("pyomo.contrib.appsi.solvers.highs", "pyomo.contrib.solver.solvers.highs"):
        if _m in sys.modules:
            sys.modules[_m].capture_output = pyomo.common.tee.capture_output
    # 2) the notebook's output returns None from write(); model.pprint()/display() need a number
    if not getattr(_fmt.StreamIndenter, "_browser_patched", False):
        class _Counted:
            def __init__(self, s): self._s = s
            def write(self, data): self._s.write(data); return len(data)
            def __getattr__(self, name): return getattr(self._s, name)
        def _init(self, ostream, indent=" " * 4, _orig=_fmt.StreamIndenter.__init__):
            _orig(self, ostream, indent)
            if not isinstance(self.target_os, _Counted):
                object.__setattr__(self, "target_os", _Counted(self.target_os))
        _fmt.StreamIndenter.__init__ = _init
        _fmt.StreamIndenter._browser_patched = True

The model as a function: `times[job]` lists the job's processing time on each machine, in machine order.
It builds the variables and the three groups of constraints above, solves with HiGHS and prints the best
order with its makespan.

In [ ]:
import pyomo.environ as pyo


def solve_flow_shop(times, title=""):
    jobs = list(times)                               # job names, e.g. "P", "Q", ...
    machines = range(len(times[jobs[0]]))            # 0 = first machine (A), 1 = B, ...
    big_h = sum(sum(t) for t in times.values())      # no schedule is longer than doing everything in a row
    pairs = [(a, b) for a in jobs for b in jobs if a < b]

    m = pyo.ConcreteModel()
    m.s = pyo.Var(jobs, machines, within=pyo.NonNegativeReals)   # start times
    m.y = pyo.Var(pairs, within=pyo.Binary)                      # y[a, b] = 1: a before b on every machine
    m.cmax = pyo.Var(within=pyo.NonNegativeReals)
    m.obj = pyo.Objective(expr=m.cmax, sense=pyo.minimize)

    m.cons = pyo.ConstraintList()
    for j in jobs:
        for k in machines:
            if k > 0:   # job j reaches machine k only after it has finished on machine k-1
                m.cons.add(m.s[j, k] >= m.s[j, k - 1] + times[j][k - 1])
        # the makespan is at least the finishing time of job j on the last machine
        m.cons.add(m.cmax >= m.s[j, machines[-1]] + times[j][machines[-1]])
    for (a, b) in pairs:
        for k in machines:   # one job at a time on machine k; y[a, b] decides which goes first
            m.cons.add(m.s[b, k] >= m.s[a, k] + times[a][k] - big_h * (1 - m.y[a, b]))
            m.cons.add(m.s[a, k] >= m.s[b, k] + times[b][k] - big_h * m.y[a, b])

    pyo.SolverFactory('appsi_highs').solve(m)

    order = sorted(jobs, key=lambda j: pyo.value(m.s[j, 0]))   # the order on the first machine
    print(f"{title}  optimal makespan = {pyo.value(m.cmax):g}   order: {' '.join(order)}")
    return pyo.value(m.cmax)


Q1 and Q2: two machines (A, B). Johnson's rule is optimal for two machines, so the solver should agree with
it.

In [ ]:
# Q1: time on machine A, time on machine B
q1 = {"P": [5, 7], "Q": [7, 5], "R": [2, 3], "S": [4, 8], "T": [7, 4]}
solve_flow_shop(q1, "Q1")

# Q2
q2 = {"P": [8, 8], "Q": [12, 15], "R": [12, 13], "S": [16, 19], "T": [7, 8]}
solve_flow_shop(q2, "Q2")


Q3: five machines (A–E). Johnson's rule is no longer guaranteed to be optimal, so the Campbell, Dudek and
Smith heuristic only gives a good order; the program finds the best one.

In [ ]:
# Q3: times on machines A, B, C, D, E
q3 = {"P": [8, 11, 3, 3, 11],
      "Q": [7, 5, 8, 5, 9],
      "R": [2, 3, 5, 7, 3],
      "S": [6, 9, 9, 14, 2],
      "T": [7, 4, 2, 12, 3]}
solve_flow_shop(q3, "Q3")


## What do you notice?

In Q3 the best order found by the heuristic (machines A and E considered, R Q P T S) has makespan 66, and the
program finds 64. Is the heuristic wrong?

<details><summary>Answer</summary>

No: with more than two machines Johnson's rule (and the Campbell, Dudek and Smith method built on it) is a
heuristic. It gives a good order quickly, but not necessarily the best one. The program checks every order
implicitly; for 5 jobs there are only $5! = 120$ orders, so you could also check them all by hand in Excel.
If you also allow each machine to process the jobs in a *different* order (one $y$ variable per pair of jobs
**and** per machine, as in the job shop model), the makespan for Q3 drops to 63: such schedules are no longer
a flow shop in the sense of the lecture.
</details>
